<a href="https://colab.research.google.com/github/abhinav77040/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhinav77040/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

I chose a Random Forest Regressor because my task is to predict a continuous score (page performance). Random Forest can model non-linear relationships between SEO features such as search volume, content age, word count, CTR, and engagement. It also provides feature importance that helps explain which signals influence predictions.

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [23]:
from sklearn.model_selection import train_test_split

features = [
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "clicks_90d"

model_df = df[features + [target]].dropna()

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(16014, 7)
(4004, 7)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [24]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [25]:
predictions = model.predict(X_test)

In [26]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE:", round(mae,2))
print("R²:", round(r2,3))

MAE: 20.46
R²: 0.123


In [27]:
baseline_prediction = np.repeat(y_train.mean(), len(y_test))

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

print("Baseline MAE:", round(baseline_mae,2))

Baseline MAE: 26.93


In [28]:
comparison = pd.DataFrame({
    "Model":["Baseline","Random Forest"],
    "MAE":[baseline_mae,mae]
})

comparison

,Model,MAE
0,Baseline,26.934921
1,Random Forest,20.458404


In [29]:
importance = pd.DataFrame({
    "Feature":features,
    "Importance":model.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
5,ctr,0.270501
2,word_count,0.254722
6,avg_position,0.180499
3,char_count,0.166305
0,search_volume,0.055145
1,competition,0.038938
4,days_since_last_update,0.033889


In [30]:
results = pd.DataFrame({
    "Actual":y_test,
    "Predicted":predictions
})

results["Error"] = abs(
    results["Actual"]-
    results["Predicted"]
)

results.sort_values(
    "Error",
    ascending=False
).head(10)

,Actual,Predicted,Error
10741,4178,200.89,3977.11
2797,2138,66.57,2071.43
14090,2032,127.20,1904.80
24993,1223,71.41,1151.59
3295,1111,187.02,923.98
15901,965,122.38,842.62
8578,912,131.97,780.03
18803,1188,498.61,689.39
5033,13,675.09,662.09
7284,735,93.30,641.70


The Random Forest model produced lower prediction error than the baseline average predictor, indicating that it captured useful relationships in the historical data.

The largest errors occurred for pages with unusually high or low traffic, suggesting that these pages may be influenced by factors not included in the current feature set, such as seasonality or external events.

The feature importance analysis suggests that search volume, days since the last update, and CTR contribute most to the model's predictions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.